<a href="https://colab.research.google.com/github/korzhimanov/dsp-seminars/blob/main/notes/7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практическое занятие №7: Адаптивная фильтрация и частотно-временной анализ

## 1. Алгоритм LMS

Для КИХ-фильтра длины $N$ на каждом отсчёте:
$$
\mathbf{x}[n] = [x[n], x[n-1], \dots, x[n-N+1]]^T, \quad
y[n] = \mathbf{w}^T\mathbf{x}[n], \quad
e[n] = d[n] - y[n], \quad
\mathbf{w} \leftarrow \mathbf{w} + \mu\,e[n]\,\mathbf{x}[n].
$$

```python
for n in range(N - 1, len(x)):
    x_vec = x[n - N + 1:n + 1][::-1]
    y = w @ x_vec
    e[n] = d[n] - y
    w = w + mu * e[n] * x_vec
```

- **Кривая обучения** – зависимость квадрата ошибки от номера шага в дБ. Квадрат ошибки сильно флуктуирует, поэтому его сглаживают: `np.convolve(e**2, np.ones(50)/50, mode='same')`.
- **Шаг адаптации:** малый $\mu$ – медленная сходимость, но малая установившаяся ошибка; большой – быстрая сходимость, но большая ошибка и риск расходимости.
- **Устойчивость:** для белого входного сигнала с дисперсией $\sigma_x^2$ приближённо $0 < \mu < \dfrac{2}{N\sigma_x^2}$ (точнее, $\mu < 2/\lambda_{\max}$, где $\lambda_{\max}$ – наибольшее собственное число корреляционной матрицы входа).

## 2. NLMS

$$
\mathbf{w} \leftarrow \mathbf{w} + \frac{\mu}{\|\mathbf{x}[n]\|^2 + \varepsilon}\,e[n]\,\mathbf{x}[n], \qquad 0 < \mu < 2.
$$
Шаг автоматически подстраивается под мощность входного сигнала; $\varepsilon$ (например, $10^{-6}$) защищает от деления на ноль.

## 3. Конфигурации адаптивного фильтра

| Задача | Вход фильтра $x$ | Желаемый сигнал $d$ | Результат |
|---|---|---|---|
| Идентификация системы | сигнал на входе системы | сигнал на выходе системы | $\mathbf{w}$ – модель системы |
| Подавление шума | опорный шум | полезный сигнал + шум | ошибка $e$ – очищенный сигнал |
| Эхокомпенсация | сигнал дальнего абонента | сигнал микрофона с эхом | ошибка $e$ – речь без эха |

## 4. Скалярный фильтр Калмана

Модель: $x_k = x_{k-1} + w_k$ (дисперсия шума процесса $Q$), $z_k = x_k + v_k$ (дисперсия шума измерений $R$).

```python
x_est, P = 0.0, 1.0
for zk in z:
    P = P + Q                    # прогноз
    K = P / (P + R)              # коэффициент Калмана
    x_est = x_est + K*(zk - x_est)
    P = (1 - K) * P              # коррекция
```

В установившемся режиме $K \to K_\infty = \dfrac{-Q + \sqrt{Q^2 + 4QR}}{2R}$, и фильтр совпадает с экспоненциальным сглаживанием с $\alpha = K_\infty$.

## 5. STFT в SciPy: класс `ShortTimeFFT`

```python
from scipy.signal import ShortTimeFFT
from scipy.signal.windows import hann

SFT = ShortTimeFFT(hann(Nw, sym=False), hop=Nw//4, fs=fs, mfft=None, scale_to=None)
X = SFT.stft(x)              # комплексные коэффициенты, форма (частоты, кадры)
S = SFT.spectrogram(x)       # |X|^2
f = SFT.f                    # частоты бинов, Гц
tt = SFT.t(len(x))           # моменты времени кадров, с
extent = SFT.extent(len(x))  # границы для plt.imshow(..., extent=extent, origin='lower', aspect='auto')
x_rec = SFT.istft(X, k1=len(x))   # обратное преобразование
```

- `hop` – шаг между окнами; перекрытие 50–75 % (`hop = Nw//2 ... Nw//4`).
- `mfft` – длина БПФ (дополнение нулями делает спектр глаже, но не улучшает разрешение).
- `scale_to='magnitude'` или `'psd'` – нормировка спектра.
- Спектрограмму показывают в дБ: `10*np.log10(S + 1e-12)`; полезно ограничить динамический диапазон (`vmin = max - 60`).
- Упрощённый вариант: `plt.specgram(x, NFFT=Nw, Fs=fs, noverlap=Nw//2)`.

**Разрешение:** по частоте $\Delta f \approx k\,f_s/N_w$ ($k \approx 2$ для окна Ханна), по времени $\Delta t \approx N_w/f_s$.

## 6. Маскирование в частотно-временной области

```python
X = SFT.stft(x)
mask = np.ones(X.shape)
band = np.abs(SFT.f - f0) < df                      # строки – частоты
frames = (SFT.t(len(x)) > t1) & (SFT.t(len(x)) < t2) # столбцы – кадры
mask[np.ix_(band, frames)] = 0
y = SFT.istft(X * mask, k1=len(x))
```

## 7. Принцип неопределённости

$$
\Delta t\,\Delta f \ge \frac{1}{4\pi},
$$
где $\Delta t$ и $\Delta f$ – среднеквадратичные ширины распределений $|w(t)|^2$ и $|W(f)|^2$. Равенство достигается только для гауссова окна. При численном расчёте используйте симметричную частотную ось (`np.fft.fftfreq` с `np.fft.fftshift`) и большую длину БПФ.

## 8. Типичные ошибки

1. **Порядок отсчётов в векторе LMS:** `x[n-N+1:n+1][::-1]` – первым должен идти самый свежий отсчёт $x[n]$, иначе коэффициенты получатся в обратном порядке.
2. **Расходимость LMS:** при слишком большом шаге ошибка растёт экспоненциально и появляются `inf`/`nan`; уменьшите `mu` или используйте NLMS.
3. **Перепутаны сигналы в схеме подавления шума:** на вход фильтра подаётся опорный шум, а не сигнал микрофона.
4. **Ось частот спектрограммы:** без `extent` или `SFT.f` по осям будут номера бинов и кадров, а не герцы и секунды.
5. **Краевые эффекты STFT:** в начале и конце записи окно выходит за её пределы; яркие полосы у краёв спектрограммы – не свойство сигнала.
6. **Стерео:** аудиофайл имеет форму `(N, 2)`; перед анализом усредните каналы.